# TaaS Model & Agentic Studio — CPU / CUDA / Ollama Quick Launch


**Hypergraph integration:** Hypergraph Explorer is part of the Studio build. Its **Run discovery** action reuses the authenticated Core discovery workflow and displays the server-generated projection. This notebook uses one checkout, one Studio process, one token and one Colab proxy; it does not install or use browser automation.
**Operator guide:** [Integrated Colab quickstart](../../docs/deployment/COLAB_T4_QUICKSTART.md)

This notebook launches the integrated Model/Agent/TaaS Studio from an immutable repository revision inside the current Colab runtime.

**Available in this pinned revision:** Hugging Face model search and metadata inspection, explicit model loading, real model-backed chat inference, model-assisted GoalSpec proposals, TaaS tool discovery and pre-execution assurance, and configured MCP stdio server/tool inspection.

**Important boundaries:** MCP inspection does not invoke tools; TaaS certification is pre-execution evidence; the Agent workspace does not yet run an autonomous tool-calling loop. Persistent run history, streaming traces, multi-user isolation and production deployment are not provided.

In [ ]:
import os, sys, subprocess, pathlib, time, shutil, urllib.request, secrets
from IPython.display import display, HTML

REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
TAAS_COMMIT = "2cc97aa699db598a479786d37d55ee8b8280923d"
WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
APP_BIND_HOST, APP_HEALTH_HOST, APP_PORT = "0.0.0.0", "127.0.0.1", 8000

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)
subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()
assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)
assert (WORKDIR / "pyproject.toml").is_file()
assert (WORKDIR / "src" / "taas").is_dir()
print("TaaS revision:", actual)

## Hardware detection

A T4 is useful for local model inference when Colab allocates one. The TaaS discovery engine itself is CPU-oriented and does not require a GPU.

The model runtime loads weights in the same Python process that hosts this Studio API. A successful GPU probe is not a model-load test; verify actual load and inference from Model Studio. If no GPU is available, CPU mode may still work for sufficiently small models.

In [ ]:
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
GPU_AVAILABLE = gpu_check.returncode == 0
print(gpu_check.stdout if GPU_AVAILABLE else "No NVIDIA GPU visible; Hugging Face CPU and Ollama remain available.")
print("Python:", sys.version.split()[0])

## Install and verify TaaS Studio

Install the Studio API and optional Hugging Face inference dependencies into the active Colab interpreter. The model runtime requires the `hf` extra (Transformers, PyTorch, Accelerate and Safetensors). If installation or import verification fails, the notebook stops with the diagnostic output.

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[studio,hf]"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(install.stdout[-4000:])
if install.returncode:
    print(install.stderr[-4000:])
    raise RuntimeError("TaaS Studio installation failed")
import_check = subprocess.run(
    [
        sys.executable, "-c",
        "import traceback, importlib; "
        "importlib.import_module('taas'); "
        "importlib.import_module('fastapi'); "
        "importlib.import_module('uvicorn'); "
        "importlib.import_module('torch'); "
        "importlib.import_module('transformers'); "
        "importlib.import_module('safetensors'); "
        "importlib.import_module('sentencepiece'); "
        "importlib.import_module('tiktoken'); "
        "importlib.import_module('taas.studio.ollama_runtime'); "
        "importlib.import_module('taas.studio.api'); "
        "print('TaaS Model/Agent Studio imports OK')"
    ],
    cwd=WORKDIR, text=True, capture_output=True
)
print(import_check.stdout)
if import_check.returncode:
    print("=== IMPORT CHECK STDERR / TRACEBACK ===")
    print(import_check.stderr[-12000:])
    print("Python executable:", sys.executable)
    print("Working directory:", WORKDIR)
    print("Package source exists:", (WORKDIR / "src" / "taas" / "__init__.py").is_file())
    raise RuntimeError("TaaS Model/Agent Studio import check failed; see traceback above")

## Optional: enable Ollama in this Colab runtime

By default, skip the next cell and use Hugging Face CPU/CUDA. To test Ollama, run the optional setup cell **before** launching Studio. It installs Ollama, starts its daemon in this VM, and pulls `qwen2.5:0.5b`. This is separate from the Hugging Face loader and can consume additional RAM/disk. See [Model Runtime Backends](../../docs/deployment/MODEL_RUNTIME_BACKENDS.md).

In [ ]:
ENABLE_OLLAMA = False  # Change to True only when you want the optional Ollama backend
if ENABLE_OLLAMA:
    if not shutil.which("ollama"):
        subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
    ollama_env = os.environ.copy()
    ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
    if "ollama_process" not in globals() or ollama_process.poll() is not None:
        ollama_log = open("/content/ollama.log", "a", buffering=1)
        ollama_process = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=ollama_log, stderr=subprocess.STDOUT)
    for attempt in range(45):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError("Ollama daemon did not become ready; inspect /content/ollama.log")
    subprocess.run(["ollama", "pull", "qwen2.5:0.5b"], check=True)
    os.environ["TAAS_OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
    print("Ollama is ready. Select Ollama in the Studio runtime panel.")
else:
    print("Ollama setup skipped. Hugging Face CPU/CUDA remains available.")


## Launch Studio

An ephemeral API token is generated automatically for this Colab runtime. Copy it into the Studio token field; it is not committed or persisted by this notebook.

The default TaaS model profile remains **strict and uncalibrated** unless an operator explicitly supplies a measured `TAAS_MODEL_PROFILE`. Installing or loading a Hugging Face model does not manufacture capability calibration. Unknown native capabilities therefore remain unassumed.

In [ ]:
api_token = secrets.token_urlsafe(32)
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ.pop("TAAS_MODEL_PROFILE", None)
print("Temporary Studio API token (copy into Studio; do not share):")
print(api_token)

# Make rerunning this cell safe, including after the Stop cell was used.
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try:
        studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.close()

studio_log = open("/content/taas-studio.log", "a", buffering=1)
studio_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "taas.studio.api:app",
     "--host", APP_BIND_HOST, "--port", str(APP_PORT)],
    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT
)

def wait_http(url, attempts=40):
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError("Studio exited; inspect /content/taas-studio.log")
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.status, response.read().decode("utf-8")
        except Exception as exc:
            last = exc
            time.sleep(1)
    raise RuntimeError(f"Studio did not become ready: {last}")

print("Health:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/health"))
print("Readiness:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/ready"))
print("Studio process PID:", studio_process.pid)


## Open TaaS Studio

Run this cell and open the link. Keep the Colab runtime connected while using Studio.

In [ ]:
from google.colab import output
if "proxy_url" not in globals() or not proxy_url:
    proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Studio</a></p>'))
print("Studio PID:", studio_process.pid)
print("Existing proxy reused; one Colab runtime and one Studio process.")


## Core-first Objective Engineering acceptance checks

This cell verifies the canonical objective endpoints, generic intent contract, exact Theta type enforcement, and Core-first UI workflow. It does not invoke tools.

After loading a model in Studio, set `RUN_LIVE_OBJECTIVE_TEST = True` to send one real prompt through model decomposition, structural validation, objective conformance and TaaS Core `gap_analysis`. A rejected proposal is reported as a failed acceptance test; it is never auto-applied or executed.


In [ ]:
import json
import urllib.request

# Ensure the repository's src-layout package is importable even if editable
# installation was skipped or the Colab runtime was restarted.
from pathlib import Path
import sys
_taas_src = Path(WORKDIR) / "src"
if not (_taas_src / "taas" / "__init__.py").is_file():
    raise RuntimeError(
        f"TaaS source tree not found at {_taas_src}. Run the first setup cell "
        "from the beginning to clone and pin the repository."
    )
if str(_taas_src) not in sys.path:
    sys.path.insert(0, str(_taas_src))
print("TaaS import source:", _taas_src)

from taas.studio.objective_contract import derive_objective_contract
from taas.studio.analyst_contract import validate_theta_types
from taas.studio.api import app

# 1) Canonical API and UI surface
paths = set(app.openapi()["paths"])
assert "/api/v1/objectives/prepare" in paths
assert "/api/v1/objectives/discover" in paths
assert "/api/v1/agents/decompose" not in paths

page = urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/").read().decode("utf-8")
assert 'runtimePost("/api/v1/objectives/prepare"' in page
assert 'fetch("/api/v1/objectives/discover"' in page
assert 'id="agent-route-prompt-core"' in page

# 2) Domain-independent objective contract smoke tests
cases = [
    ("Find the latest research papers about federated learning.", "search"),
    ("Summarise the document supplied in the prompt.", "summarise"),
    ("Translate this paragraph into French.", "translate"),
    ("Count all invoices issued this year.", "count"),
    ("Calculate average sales by region.", "aggregate"),
    ("Compare these two provided architectures.", "compare"),
    ("Identify anomalies in the supplied sensor data.", "classify"),
    ("Create a report from several external sources.", "create_document"),
    ("Generate an image illustrating the architecture.", "render_media"),
    ("Send an email notification to the project owner.", "send"),
]
contracts = {}
for prompt, expected_class in cases:
    contract = derive_objective_contract(prompt)
    contracts[prompt] = contract
    assert expected_class in contract["required_goal_classes"], (prompt, contract)
    print(expected_class, "OK:", prompt)

# Distinguish supplied content from external-source work. This prevents
# unnecessary retrieval/discovery for local transformations and comparisons.
for prompt in (
    "Summarise the document supplied in the prompt.",
    "Translate this paragraph into French.",
    "Compare these two provided architectures.",
    "Identify anomalies in the supplied sensor data.",
):
    contract = contracts[prompt]
    assert contract["user_supplied_content"] is True, (prompt, contract)
    assert contract["source_data_required"] is False, (prompt, contract)

for prompt in (
    "Find the latest research papers about federated learning.",
    "Count all invoices issued this year.",
    "Calculate average sales by region.",
    "Create a report from several external sources.",
):
    contract = contracts[prompt]
    assert contract["source_data_required"] is True, (prompt, contract)

print("Supplied-content vs external-source routing checks OK")

# 2b) Objective Contract v2: versioned outcomes and scenario-independent frames.
# Each lexical outcome is a proposal for Core validation, not a semantic proof.
for prompt, expected in cases:
    frame = contracts[prompt]
    assert frame["contract_version"] == "2.0", (prompt, frame)
    assert [item["goal_class"] for item in frame["outcomes"]] == frame["required_goal_classes"]
    assert all(item["required"] for item in frame["outcomes"])
    assert frame["review"]["semantic_correctness_verified"] is False
    assert "inputs" in frame["constraints"] and "scope" in frame["constraints"]

compound = derive_objective_contract(
    "Search for recent papers on formal concept analysis, compare the findings, "
    "synthesise the evidence, and create a report."
)
assert [item["goal_class"] for item in compound["outcomes"]] == [
    "search", "compare", "synthesise", "create_document"
], compound
assert compound["constraints"]["inputs"]["source_data_required"] is True
assert compound["review"]["semantic_correctness_verified"] is False
print("Objective Contract v2 compound-outcome checks OK:", [
    item["goal_class"] for item in compound["outcomes"]
])

# Run the Objective Contract v2 and GoalSpec conformance suites against the exact pinned checkout.
v2_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_objective_contract_v2.py", "tests/test_goalspec_conformant_agent.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(v2_tests.stdout)
if v2_tests.returncode:
    print(v2_tests.stderr[-8000:])
    raise RuntimeError("Objective Contract v2 pytest acceptance failed")


french = derive_objective_contract("Compter toutes les factures émises cette année.")
assert french["operation"] == "count"
assert french["temporal_scope"]["expression"] == "this_year"
assert french["exactness_required"] == "exact"

grouped = derive_objective_contract("Calculate average sales by region.")
assert grouped["grouped_result_required"] is True
assert grouped["numeric_result_required"] is False

# 3) Exact Theta vocabulary enforcement
invalid_type = {
    "prompt_inputs": ["text"],
    "goals": [{
        "id": "g1",
        "class": "retrieve",
        "in": ["query"],
        "out": "record:invoice[]",
        "desc": "Retrieve invoice records",
    }],
}
theta_error = validate_theta_types(invalid_type)
assert theta_error and "undeclared" in theta_error
print("Theta rejects undeclared type combinations OK")

# 4) Metadata-only tool discovery through TaaS Core. No MCP invocation occurs.
DISCOVERY_SMOKE_TEST = True
if DISCOVERY_SMOKE_TEST:
    token = os.environ["TAAS_STUDIO_API_TOKEN"]
    discovery_payload = json.dumps({
        "human_review_confirmed": True,
        "goal_spec": {
            "prompt": "Retrieve records matching the provided query from the configured source.",
            "prompt_inputs": ["text"],
            "goals": [{
                "id": "g1",
                "class": "retrieve",
                "in": ["query"],
                "out": "record[]",
                "eff": ["read(sample_registry)"],
                "sigma": "data",
                "n": 1,
                "exactness": "tolerant",
                "deps": [],
                "desc": "Retrieve records matching a query from the configured source.",
            }],
        },
        "tools": [{
            "server": "sample_registry",
            "name": "lookup_records",
            "description": "Search and return records matching a query.",
            "input_schema": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        }],
    }).encode("utf-8")
    req = urllib.request.Request(
        f"http://127.0.0.1:{APP_PORT}/api/v1/objectives/discover",
        data=discovery_payload,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    discovery_result = json.loads(urllib.request.urlopen(req).read().decode("utf-8"))
    assert discovery_result["status"] in {"selected", "abstained"}
    assert discovery_result["execution_performed"] is False
    print("Core discovery smoke test:", discovery_result["status"])
    print("Selected tools:", discovery_result["selected_tools"])
    print("Hypergraph projection:", discovery_result["hypergraph_projection_status"])

# 5) Optional real-model -> GoalSpec -> Core gap-analysis test
RUN_LIVE_OBJECTIVE_TEST = False
if RUN_LIVE_OBJECTIVE_TEST:
    token = os.environ["TAAS_STUDIO_API_TOKEN"]
    request = urllib.request.Request(
        f"http://127.0.0.1:{APP_PORT}/api/v1/models/runtime",
        headers={"Authorization": f"Bearer {token}"},
    )
    runtime = json.loads(urllib.request.urlopen(request).read().decode("utf-8"))
    ollama_selected = runtime.get("providers", {}).get("ollama", {}).get("model_selected")
    provider = "ollama" if ollama_selected else "huggingface"
    loaded = ollama_selected if provider == "ollama" else runtime.get("model_loaded", False)
    if not loaded:
        print("SKIPPED: load a model in Studio, then set RUN_LIVE_OBJECTIVE_TEST=True.")
    else:
        payload = json.dumps({
            "provider": provider,
            "prompt": "Explain the difference between precision and recall.",
            "max_new_tokens": 1200,
        }).encode("utf-8")
        req = urllib.request.Request(
            f"http://127.0.0.1:{APP_PORT}/api/v1/objectives/prepare",
            data=payload,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        result = json.loads(urllib.request.urlopen(req).read().decode("utf-8"))
        print(json.dumps({
            "workflow": result.get("workflow"),
            "structural_validation": result.get("structural_validation"),
            "conformance": result.get("conformance"),
            "core_analysis": result.get("core_analysis"),
            "diagnostics": result.get("semantic_diagnostics"),
        }, indent=2))
        assert result.get("workflow", {}).get("objective_interpretation") == "completed"
        assert result.get("conformance", {}).get("semantic_correctness_verified") is False
        if result.get("workflow", {}).get("core_gap_analysis") == "not_analyzed":
            print("LIVE OBJECTIVE TEST: NOT ACCEPTED — inspect diagnostics and attempt history.")
        else:
            print("LIVE OBJECTIVE TEST: Core gap_analysis executed; human review is still required.")
else:
    print("Live inference test is optional and currently disabled. Load a model first.")


## Stop Studio

Run this cell when finished. Colab storage and runtime memory are temporary.

In [ ]:
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try:
        studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.flush()
    studio_log.close()
if "ollama_process" in globals() and ollama_process.poll() is None:
    ollama_process.terminate()
    try:
        ollama_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        ollama_process.kill()
        ollama_process.wait(timeout=5)
print("TaaS Studio and optional Ollama daemon stopped.")
